## Clean and Export `00_Datasets.txt` to CSV

In [1]:
import json
import re
import pandas as pd

# Load data — handle BOM, control chars, and truncation
with open('Karin_dataset/00_Datasets.txt', 'r', encoding='utf-8-sig') as f:
    raw = f.read().strip()

# Remove invalid control characters (keep \n, \r, \t)
raw = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', raw)

# Fix truncated JSON
if raw.endswith(','):
    raw = raw[:-1] + '\n]'
elif not raw.endswith(']'):
    raw = raw + '\n]'

data = json.loads(raw, strict=False)
df = pd.DataFrame(data)
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Loaded: 95904 rows, 21 columns


,num_0,num_1,num_r,thread_id,thread_name,user_name,user_id,messages,length,helpful,...,tag,datetime,profile_like,emo_like,user_type,user_sameIP,IP,IP_check,link,messages_ref
0,0,0,0,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,สมาชิกหมายเลข 7464651,7464651,ก้าวไกลคือแบบอย่างการเมืองสุจริต ไม่ซื้อเสียงจ...,154,0,...,[การเมือง],07/10/2023 09:19:04,"[4849556, 486114, 5038844]","[ขำกลิ้ง, ขำกลิ้ง, ขำกลิ้ง]",registered,"[7450328, 7242702, 7452177, 7460975]","[58.136.243.156, 58.136.245.118]",True,[7464651],ก้าวไกลคือแบบอย่างการเมืองสุจริต ไม่ซื้อเสียงจ...
1,1,0,1,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,สมาชิกหมายเลข 7361167,7361167,พปชร รทสช ต่างหากที่เป็นพรรคสุจริต ถ้าทำแบบนั้...,80,0,...,[การเมือง],07/10/2023 09:34:22,[],[],sms,[],[],False,[7464651],พปชร รทสช ต่างหากที่เป็นพรรคสุจริต ถ้าทำแบบนั้...
2,2,0,2,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,สมาชิกหมายเลข 7576977,7576977,ขึ้น ค่าแรง 450 ทันที สูงอายุ 3000 ติดเตียง 10...,119,0,...,[การเมือง],07/10/2023 09:35:04,"[2108987, 5038844]","[ถูกใจ, ถูกใจ]",registered,[2749253],"[49.228.98.29, 27.55.68.147, 58.11.70.24, 61.7...",True,[7464651],ขึ้น ค่าแรง 450 ทันที สูงอายุ 3000 ติดเตียง 10...
3,3,0,3,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,Mr.Green,88374,ยังไม่เริ่มทำงานจริงเลย ยังพูดไม่ได้หรอก ตัวอย...,120,0,...,[การเมือง],07/10/2023 10:52:21,[],[],id_card,[],[],False,[7464651],ยังไม่เริ่มทำงานจริงเลย ยังพูดไม่ได้หรอก ตัวอย...
4,4,0,4,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,Nnkw,486114,ไม่ซื้อด้วยเสียง แต่เข้ามาด้วยกระบวนการพิเศษขอ...,422,1,...,[การเมือง],07/10/2023 11:50:35,"[663291, 181908]","[ถูกใจ, ถูกใจ]",id_card,[],[],False,[7464651],ไม่ซื้อด้วยเสียง แต่เข้ามาด้วยกระบวนการพิเศษขอ...


In [2]:
# 1. Filter rows where tag contains "การเมือง"
df = df[df['tag'].apply(lambda x: 'การเมือง' in x if isinstance(x, list) else False)].copy()
print(f'After tag filter: {df.shape[0]} rows')

After tag filter: 13482 rows


In [3]:
# 2. Filter only users that have IP_check
df = df[df['IP_check'] == True].copy()
print(f'After IP_check filter: {df.shape[0]} rows')

After IP_check filter: 5457 rows


In [4]:
# 3. Create is_sockpuppet: 1 if user_sameIP is a non-empty list, else 0
df['is_sockpuppet'] = df['user_sameIP'].apply(
    lambda x: 1 if isinstance(x, list) and len(x) > 0 else 0
)
print(f'is_sockpuppet distribution:\n{df["is_sockpuppet"].value_counts()}')

is_sockpuppet distribution:
is_sockpuppet
0    3067
1    2390
Name: count, dtype: int64


In [5]:
# 3b. Create sockpuppet_group_id for stratified splitting later
# Before dropping user_sameIP, build groups
import networkx as nx

# Build a graph where edges = IP-sharing
G = nx.Graph()
sock_df = df[df['is_sockpuppet'] == 1].drop_duplicates('user_id')

for _, row in sock_df.iterrows():
    uid = row['user_id']
    G.add_node(uid)
    same_ip_users = row['user_sameIP'] if isinstance(row['user_sameIP'], list) else []
    for other in same_ip_users:
        G.add_edge(uid, other)

# Connected components = sockpuppet groups
user_to_group = {}
for group_id, component in enumerate(nx.connected_components(G)):
    for user in component:
        user_to_group[user] = group_id

df['sockpuppet_group_id'] = df['user_id'].map(user_to_group).fillna(-1).astype(int)
print(f'Number of sockpuppet groups: {len(set(user_to_group.values()))}')

Number of sockpuppet groups: 74


In [6]:
# 4. Rename columns
df = df.rename(columns={
    'thread_name': 'topic',
    'profile_like': 'user_react'
})

# 5. Drop unused columns
drop_cols = ['num_r', 'user_name', 'forum', 'helpful', 'emo_like', 'tag'
             'user_type', 'user_sameIP', 'IP', 'IP_check', 'link', 'messages_ref']
            # NOTE: If we filter row with only tag contains "การเมือง", we can drop colum 'tag'
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 6. Reorder: thread_id first, topic second, then the rest
cols = df.columns.tolist()
cols.remove('thread_id')
cols.remove('topic')
df = df[['thread_id', 'topic'] + cols]

print(f'Final: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
df.head()

Final: 5457 rows, 13 columns
Columns: ['thread_id', 'topic', 'num_0', 'num_1', 'user_id', 'messages', 'length', 'tag', 'datetime', 'user_react', 'user_type', 'is_sockpuppet', 'sockpuppet_group_id']


,thread_id,topic,num_0,num_1,user_id,messages,length,tag,datetime,user_react,user_type,is_sockpuppet,sockpuppet_group_id
0,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,0,0,7464651,ก้าวไกลคือแบบอย่างการเมืองสุจริต ไม่ซื้อเสียงจ...,154,[การเมือง],07/10/2023 09:19:04,"[4849556, 486114, 5038844]",registered,1,0
2,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,2,0,7576977,ขึ้น ค่าแรง 450 ทันที สูงอายุ 3000 ติดเตียง 10...,119,[การเมือง],07/10/2023 09:35:04,"[2108987, 5038844]",registered,1,1
7,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,7,0,7450328,ปชป.แมงสาบ 2,12,[การเมือง],07/10/2023 13:36:40,[],registered,1,0
8,42111000,ก้าวไกลคือแบบอย่างการเมืองสุจริต,8,0,4130679,ก้าวไกลคือแบบอย่างการใช้ม๊อบบ่อนทำลายรัฐบาลเก่...,237,[การเมือง],07/10/2023 15:08:07,[],registered,0,-1
40,42111007,การอ้าง 14 ล้านเสียง ว่าคือ ฉันทามติ ให้ส้ม แก...,0,0,7585251,การอ้าง 14 ล้านเสียง ว่าคือ ฉันทามติ ให้ส้ม แก...,949,"[พรรคการเมือง, การเมือง, นักการเมือง, เลือกตั้...",07/10/2023 09:21:37,"[718696, 7439140, 6368036]",registered,0,-1


In [7]:
# 7. Summary
total_rows = len(df)
unique_threads = df['thread_id'].nunique()
unique_users = df['user_id'].nunique()
sockpuppet_users = df[df['is_sockpuppet'] == 1]['user_id'].nunique()
normal_users = df[df['is_sockpuppet'] == 0]['user_id'].nunique()
sock_rows = df['is_sockpuppet'].sum()

print('=== Dataset Summary ===')
print(f'Total rows:          {total_rows:,}')
print(f'Unique threads:      {unique_threads:,}')
print(f'Unique users:        {unique_users:,}')
print(f'  - Sockpuppet:      {sockpuppet_users:,}')
print(f'  - Normal:          {normal_users:,}')
print(f'Sockpuppet rows:     {sock_rows:,} ({sock_rows/total_rows*100:.1f}%)')
print(f'Normal rows:         {total_rows - sock_rows:,} ({(total_rows - sock_rows)/total_rows*100:.1f}%)')

=== Dataset Summary ===
Total rows:          5,457
Unique threads:      974
Unique users:        394
  - Sockpuppet:      142
  - Normal:          252
Sockpuppet rows:     2,390 (43.8%)
Normal rows:         3,067 (56.2%)


In [8]:
# 8. Export to CSV
output_filename = '00_Datasets_After_preprocessed.csv'

df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f'Exported to {output_filename}')

Exported to 00_Datasets_After_preprocessed.csv
